In [ ]:
import pandas as pd
import numpy as np
import os
# قرائة الملف
data = pd.read_csv('data/dirty_cafe_sales.csv')

df = pd.DataFrame(data)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [ ]:
# تحول اسامي الاعمده
df = df.rename(columns={
    'Transaction ID': 'Transaction_ID',
    'Total Spent': 'Total_Spent',
    'Transaction Date': 'Transaction_Date',
    'Price Per Unit': 'Price_Per_Unit',
    'Payment Method': 'Payment_Method'
    
})



In [185]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction_ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price_Per_Unit    9821 non-null   object
 4   Total_Spent       9827 non-null   object
 5   Payment_Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction_Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [ ]:
# تحويل انواع الاعمده عشان اقدر اسوي عليها تعديلات وعمليات حسابيه وكمان حق التاريخ عشان اقدر انظفه
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price_Per_Unit"] = pd.to_numeric(df["Price_Per_Unit"], errors="coerce")
df["Total_Spent"] = pd.to_numeric(df["Total_Spent"], errors="coerce")

df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")

In [229]:
df.isna().sum()

Transaction_ID      0
Item                0
Quantity            0
Price_Per_Unit      0
Total_Spent         0
Payment_Method      0
Location            0
Transaction_Date    0
dtype: int64

In [ ]:
# حذف التكرارات
df.drop_duplicates(inplace=True)

# Transaction_ID :

In [188]:
df["Transaction_ID"].duplicated().sum()

np.int64(0)

# Item :

In [219]:
df['Item'].value_counts()

Item
Juice       1120
Coffee      1116
Salad       1095
Cake        1077
Sandwich    1066
Smoothie    1042
Cookie      1028
Tea         1020
Name: count, dtype: int64

In [220]:

df["Item"] = df["Item"].replace(["ERROR", "UNKNOWN"], np.nan)

In [228]:
df.groupby("Item")["Price_Per_Unit"].unique()

Item
Cake        [3.0]
Coffee      [2.0]
Cookie      [1.0]
Juice       [3.0]
Salad       [5.0]
Sandwich    [4.0]
Smoothie    [4.0]
Tea         [1.5]
Name: Price_Per_Unit, dtype: object

In [222]:
df["Item"].isna().sum()

np.int64(921)

In [ ]:
# حدت القيم التي معاها نفس القيمه
price_item_groups = (
    df[df["Item"].notna()]
    .groupby("Price_Per_Unit")["Item"]
    .unique()
)
price_item_groups

Price_Per_Unit
1.0                [Cookie]
1.5                   [Tea]
2.0                [Coffee]
3.0           [Cake, Juice]
4.0    [Smoothie, Sandwich]
5.0                 [Salad]
Name: Item, dtype: object

In [ ]:
# يقسم القيم الفارغه بين المنتجات المتشابه نص بنص 
#  يعني مثلا اذا كان [Cake, Juice] يقوم يقسم القيم الفارغه بينهم بالنص
# لاننا لاحضت ان القيم المتكرره متاشبه حق كل منتج 
for price, items in price_item_groups.items():
    
    mask = (df["Item"].isna()) & (df["Price_Per_Unit"] == price)
    
    if len(items) > 1:
        df.loc[mask, "Item"] = np.random.choice(
            items,
            size=mask.sum()
        )
    else:
        df.loc[mask, "Item"] = items[0]

# Location :

In [192]:
df['Location'].value_counts()

Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [193]:
df['Location'] = df['Location'].replace('ERROR', 'Takeaway')
df['Location'] = df['Location'].replace('UNKNOWN', 'In-store')

In [194]:
# حساب النسب الحالية
location_distribution = df["Location"].value_counts(normalize=True)

location_distribution

Location
Takeaway    0.501856
In-store    0.498144
Name: proportion, dtype: float64

In [195]:
# تعبئة القيم المفقودة بنفس النسب
df.loc[df["Location"].isna(), "Location"] = np.random.choice(
    location_distribution.index,
    size=df["Location"].isna().sum(),
    p=location_distribution.values
)

# Transaction_Date : 

In [197]:
df["Transaction_Date"].isna().sum()

np.int64(460)

In [198]:
df = df.dropna(subset=["Transaction_Date"])

# Total_Spent :

In [ ]:
# حساب الاجمالي عن طريق ضرب الكميه في سعر الوحده
df.loc[df["Total_Spent"].isna(), "Total_Spent"] = (
    df["Quantity"] * df["Price_Per_Unit"]
)


In [201]:
df = df.dropna(subset=["Total_Spent"])

# Payment_Method :

In [204]:
df['Payment_Method'].value_counts()

Payment_Method
Digital Wallet    2188
Credit Card       2163
Cash              2148
Name: count, dtype: int64

In [203]:
df["Payment_Method"] = df["Payment_Method"].replace(["ERROR", "UNKNOWN"], np.nan)

In [ ]:
# اطلع نسبه كل متغير كم منه عشان لمن اعبي القيم الفارغه اعبيها بنسب صحيحه 
Payment_Method_distribution = df["Payment_Method"].value_counts(normalize=True)

Payment_Method_distribution

Payment_Method
Digital Wallet    0.336667
Credit Card       0.332820
Cash              0.330512
Name: proportion, dtype: float64

In [206]:
# تعبئة القيم المفقودة بنفس النسب
df.loc[df["Payment_Method"].isna(), "Payment_Method"] = np.random.choice(
    Payment_Method_distribution.index,
    size=df["Payment_Method"].isna().sum(),
    p=Payment_Method_distribution.values
)

# Quantity :

In [209]:
(df["Price_Per_Unit"] == 0).sum()

np.int64(0)

In [ ]:
# حساب الكميه عن طريق قسمه الاجمالي على سعر الوحده
df.loc[df["Quantity"].isna(), "Quantity"] = (
    df["Total_Spent"] / df["Price_Per_Unit"]
)

In [213]:
df = df.dropna(subset=["Quantity"])

# Price_Per_Unit :

In [ ]:
# حساب سعر الوحده عن طريق قسمه الاجمالي على الكميه
df.loc[df["Price_Per_Unit"].isna(), "Price_Per_Unit"] = (
    df["Total_Spent"] / df["Quantity"]
)

# save cleaned data :

In [ ]:
df.to_csv('data/cleaned_cafe_sales.csv', index=False)